# StopTheCap - E2-C1 encoder training (Colab, single T4)

Fine-tunes **distilbert-base-cased** on `datasets/canonical/e2c1.jsonl`
(23,774 rows, 1:1 human/machine), selects the decision threshold on `val` at a
5% human false-positive budget, freezes it, and evaluates `val`, `test_holdout`,
`hc3` and `raid`.

This is an **authorised training run**, not a smoke gate. Do not stop early.
The run log, evaluation JSON/Markdown, predictions and `last.pt` checkpoints are
written to Drive so a disconnect can resume.

What it does not do: no data rebuild, no threshold tuning on test splits, no
public upload, no write under `datasets/`.


## 1. Configure

In [ ]:
import os
from pathlib import Path

ZIP_PATH = "/content/drive/MyDrive/StopTheCap_E2C1.zip"
REPO_DIR = "/content/StopTheCap"
REPORT_DIR = "/content/drive/MyDrive/StopTheCap_reports/E2C1"
REPORT_PARENT = "/content/drive/MyDrive/StopTheCap_reports"

# Pinned contract values for the committed E2-C1 corpus.
CORPUS_SHA256 = "767853403bb77d72bbfa8a2a3e5dfd424ddfa5ebe9626e4cea3a6b1ccba36675"
SPLIT_DIGEST = "adfe9df5900dac1342ff241ef1eff7ed5cf933cdcc728dfe4e69e0c00d94fb97"
TOTAL_SAMPLES = 23774
TRAIN_SAMPLES = 18688

MODEL_NAME = "distilbert-base-cased"
MAX_LENGTH = 512
BATCH_SIZE = 16
GRAD_ACCUM = 2
EPOCHS = 3

print("ZIP_PATH      :", ZIP_PATH)
print("REPO_DIR      :", REPO_DIR)
print("REPORT_DIR    :", REPORT_DIR)
print("corpus sha256 :", CORPUS_SHA256)
print("split digest  :", SPLIT_DIGEST)

## 2. Mount Drive

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
print("Drive mounted at /content/drive")

## 3. Load the private project ZIP

In [ ]:
# Changing the runtime type restarts the kernel and wipes the config cell.
if "LOAD_METHOD" not in globals():
    import os
    LOAD_METHOD = "zip"
    ZIP_PATH = "/content/drive/MyDrive/StopTheCap_E2C1.zip"
    REPO_DIR = "/content/StopTheCap"
    print("NOTE: config cell had not run; using default paths.")

import os, shutil, sys, zipfile
from pathlib import Path

REQUIRED = [
    "ml/__init__.py",
    "ml/train/train_e2c1.py",
    "ml/train/e2c1_config.yaml",
    "ml/ingest/__init__.py",
    "ml/ingest/normalise.py",
    "ml/e2c1/__init__.py",
    "ml/data/__init__.py",
    "datasets/canonical/e2c1.jsonl",
    "datasets/canonical/hc3.jsonl",
    "datasets/canonical/raid.jsonl",
    "datasets/manifests/e2_c1_training.json",
    "datasets/manifests/e2_c1_corpus.json",
    "notebooks/colab_train_e2c1.ipynb",
]
EXTRACT_TMP = "/content/_stc_extract"   # overridden in tests


def _fail(msg):
    raise RuntimeError(msg)


def load_from_zip(zip_path, dest):
    z = Path(zip_path)
    if not z.is_file():
        _fail(
            "ZIP not found: " + str(z) + "\n"
            "Upload the project ZIP to Google Drive, then set ZIP_PATH.\n"
            "Build it with: python scripts/make_colab_zip.py --with-data --with-e2c1-training"
        )
    dest = Path(dest)
    tmp = Path(EXTRACT_TMP)
    for p in (dest, tmp):
        if p.exists():
            shutil.rmtree(p)
    tmp.mkdir(parents=True)

    with zipfile.ZipFile(z) as zf:
        names = zf.namelist()
        print("zip entries:", len(names))
        for n in names[:12]:
            print("   ", n)
        if not names:
            _fail("The ZIP is empty.")
        zf.extractall(tmp)

    entries = [p for p in tmp.iterdir() if p.name != "__MACOSX"]
    root = entries[0] if len(entries) == 1 and entries[0].is_dir() else tmp
    shutil.move(str(root), str(dest))
    if tmp.exists():
        shutil.rmtree(tmp, ignore_errors=True)
    return dest


if LOAD_METHOD == "zip":
    REPO = load_from_zip(ZIP_PATH, REPO_DIR)
elif LOAD_METHOD == "git":
    _fail("This private notebook loads the Drive ZIP only; set LOAD_METHOD = 'zip'.")
else:
    _fail("Unknown LOAD_METHOD " + repr(LOAD_METHOD) + ". Use 'zip'.")

REPO = Path(REPO).resolve()
missing = [f for f in REQUIRED if not (REPO / f).is_file()]
if missing:
    _fail("Project structure looks wrong; missing: " + ", ".join(missing))

os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

print()
print("project root :", REPO)
print("required     :", len(REQUIRED), "present")

## 4. Validate the E2-C1 corpus and manifests

In [ ]:
import hashlib, json
from collections import Counter

CORPUS_PATH = REPO / "datasets/canonical/e2c1.jsonl"
TRAINING_MANIFEST_PATH = REPO / "datasets/manifests/e2_c1_training.json"
CORPUS_MANIFEST_PATH = REPO / "datasets/manifests/e2_c1_corpus.json"

digest = hashlib.sha256(CORPUS_PATH.read_bytes()).hexdigest()
assert digest == CORPUS_SHA256, f"corpus sha256 {digest} != pinned {CORPUS_SHA256}"

training_manifest = json.loads(TRAINING_MANIFEST_PATH.read_text(encoding="utf-8"))
corpus_manifest = json.loads(CORPUS_MANIFEST_PATH.read_text(encoding="utf-8"))

assert training_manifest["corpus_sha256"] == CORPUS_SHA256
assert training_manifest["counts"]["total"] == TOTAL_SAMPLES
assert training_manifest["counts"]["by_label"]["0"] == training_manifest["counts"]["by_label"]["1"]

# The split digest is the one recorded when the prompt split was frozen.
split_digest = corpus_manifest["split_digest"]
assert split_digest == SPLIT_DIGEST, f"split digest {split_digest} != pinned {SPLIT_DIGEST}"
assert len(split_digest) == 64 and all(c in "0123456789abcdef" for c in split_digest)

rows = [json.loads(line) for line in CORPUS_PATH.read_text(encoding="utf-8").splitlines() if line.strip()]
partition_counts = Counter(row["partition"] for row in rows)
label_counts = Counter((row["partition"], row["label"]) for row in rows)
assert len(rows) == TOTAL_SAMPLES, len(rows)
assert partition_counts["train"] == TRAIN_SAMPLES
for partition in ("train", "val", "test_holdout"):
    assert partition_counts[partition] == training_manifest["counts"]["by_partition"][partition]
    assert label_counts[(partition, 0)] == label_counts[(partition, 1)], partition

print("corpus sha256 :", digest)
print("split digest  :", split_digest)
print("rows          :", len(rows))
print("partitions    :", dict(partition_counts))
print("labels        :", dict(Counter(row["label"] for row in rows)))

## 5. GPU / CUDA / VRAM pre-flight

In [ ]:
import math
import torch

assert torch.cuda.is_available(), (
    "CUDA is required. Runtime > Change runtime type > T4 GPU."
)
props = torch.cuda.get_device_properties(0)
vram_gb = props.total_memory / 1e9
capability = torch.cuda.get_device_capability(0)
print(f"GPU        : {props.name}")
print(f"VRAM       : {vram_gb:.2f} GB (T4 class ~15.6 GB)")
print(f"capability : sm_{capability[0]}{capability[1]}")
assert props.total_memory >= 14.5e9, f"need >= 14.5 GB VRAM, found {vram_gb:.2f} GB"

micro_batches = math.ceil(TOTAL_SAMPLES / BATCH_SIZE)
steps_per_epoch = math.ceil(micro_batches / GRAD_ACCUM)
total_steps = steps_per_epoch * EPOCHS
# 23,774 / 16 = 1,486 micro-batches; / 2 = 743 optimizer steps/epoch; x3 = 2,229.
train_micro = math.ceil(TRAIN_SAMPLES / BATCH_SIZE)
train_steps_per_epoch = math.ceil(train_micro / GRAD_ACCUM)
train_total_steps = train_steps_per_epoch * EPOCHS
# The actual train partition: 18,688 / 16 = 1,168 micro-batches; 584/epoch; 1,752 total.
print(f"all rows      : {TOTAL_SAMPLES} / ({BATCH_SIZE} x {GRAD_ACCUM}) -> "
      f"{steps_per_epoch} steps/epoch, {total_steps} total over {EPOCHS} epochs")
print(f"train split   : {TRAIN_SAMPLES} / ({BATCH_SIZE} x {GRAD_ACCUM}) -> "
      f"{train_steps_per_epoch} steps/epoch, {train_total_steps} total")

## 6. Dependencies

In [ ]:
# Colab already ships a CUDA torch build; install the rest without replacing it.
%pip install -q "torch>=2.0" "transformers>=4.40" "accelerate>=0.30" "scikit-learn>=1.3"

import accelerate
import sklearn
import torch
import transformers
import yaml

print("torch        :", torch.__version__)
print("cuda         :", torch.version.cuda)
print("transformers :", transformers.__version__)
print("accelerate   :", accelerate.__version__)
print("scikit-learn :", sklearn.__version__)
print("PyYAML       :", yaml.__version__)

## 7. Train (authorised run)

In [ ]:
import subprocess, sys, time
from pathlib import Path

REPORT_DIR = Path(REPORT_DIR)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

# Exact run: python -m ml.train.train_e2c1 --device cuda --out /content/drive/MyDrive/StopTheCap_reports/E2C1
command = [
    sys.executable, "-m", "ml.train.train_e2c1",
    "--config", str(REPO / "ml/train/e2c1_config.yaml"),
    "--corpus", str(REPO / "datasets/canonical/e2c1.jsonl"),
    "--out", str(REPORT_DIR),
    "--device", "cuda",
    "--max-length", str(MAX_LENGTH),
]
print("running:", " ".join(command), flush=True)
began = time.perf_counter()
completed = subprocess.run(command, cwd=str(REPO), text=True, capture_output=True)
wall_minutes = (time.perf_counter() - began) / 60
print(completed.stdout)
if completed.returncode != 0:
    print(completed.stderr)
    raise RuntimeError(f"training failed with exit {completed.returncode}")
print(f"training wall time: {wall_minutes:.1f} min")

## 8. Copy reports to Drive

In [ ]:
import shutil
from pathlib import Path

REPORT_DIR = Path(REPORT_DIR)
REPORT_PARENT = Path(REPORT_PARENT)
REPORT_PARENT.mkdir(parents=True, exist_ok=True)

ARTIFACTS = ["E2C1.json", "E2C1_eval.json", "E2C1_eval.md", "E2C1_predictions.jsonl"]
copied = []
for name in ARTIFACTS:
    source = REPORT_DIR / name
    assert source.is_file(), f"missing report: {source}"
    target = REPORT_PARENT / name
    shutil.copy2(source, target)
    copied.append(target)

checkpoint = REPORT_DIR / "checkpoints/last.pt"
print("reports on Drive:")
for path in copied:
    print("  ", path, path.stat().st_size, "bytes")
print("checkpoint      :", checkpoint, checkpoint.is_file())

## 9. Results

In [ ]:
import json
from pathlib import Path

eval_result = json.loads((Path(REPORT_DIR) / "E2C1_eval.json").read_text(encoding="utf-8"))
threshold = eval_result["frozen_threshold"]
print(f"frozen threshold: {threshold:.6f}")
print(f"target human FPR: {eval_result['target_fpr']:.0%}")
print()
header = f"{'split':<13} {'n':>6} {'human':>6} {'machine':>7} {'ROC-AUC':>8} {'PR-AUC':>7} {'human FPR':>10} {'recall':>7} {'ECE':>6}"
print(header)
print("-" * len(header))
for name in ("val", "test_holdout", "hc3", "raid"):
    report = eval_result["splits"].get(name)
    if report is None:
        continue
    auc = "n/a" if report["roc_auc"] is None else f"{report['roc_auc']:.3f}"
    prauc = "n/a" if report["pr_auc"] is None else f"{report['pr_auc']:.3f}"
    fpr = "n/a" if report["human_fpr"] is None else f"{report['human_fpr']:.3f}"
    print(f"{name:<13} {report['n']:>6} {report['n_human']:>6} {report['n_machine']:>7} "
          f"{auc:>8} {prauc:>7} {fpr:>10} {report['recall']:>7.3f} {report['ece']:>6.3f}")

print()
print("per-generator (test_holdout):")
for generator, values in eval_result["per_generator"].items():
    print(f"  {generator:<20} n={values['n']:<5} detection_rate={values['detection_rate']:.3f}")

print()
print("artefacts:")
for name in ("E2C1.json", "E2C1_eval.json", "E2C1_eval.md", "E2C1_predictions.jsonl"):
    path = Path(REPORT_DIR) / name
    print("  ", path, path.is_file())
print("Training complete; reports are on Drive.")